# Pacman con Redes Neuronales + AlphaBeta

## Autores: Ismael Escribano Orts y Marcos Hernández Juárez

### 1. *Introducción*

El objetivo de esta práctica se divide en dos partes, aunque ámbas relacionadas con poder jugar a Pac-Man de manera autónoma y eficiente:

1. Entrenamiento de la Red Neuronal (NN): Se ha necesitado jugar manualmente partidas para posteriormente entrenar una red neuronal que nos permita jugar de manera automática, aunque sin seguir ninguna estrategia, simplemente aprendiendo de nuestros moviemientos para después replicar nuestro comportamiento en el juego.

2. Implementación de estrategia: Adicionamente, es necesario dotar al agente de estrategia, para ello se usaría el algoritmo Minimax, pero debido a su coste computacional, utilizaremos la versión de AlphaBeta. Esto nos va a permitir añadirle un mejor criterio a la red y conseguir mejores resultados globales.

### 2. *Nuevas heurísticas*

#### 2.1. *Heurísticas iniciales*
Tanto la NN como el algoritmo AlphaBeta van a trabajar según unas heurísticas dadas, que será nuestra estrategia, en el código dado tenemos las siguientes heurísticas iniciales:
```python
# Factor 1: Distancia a la comida más cercana
if food:
    min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
    score += 1.0 / (min_food_distance + 1)

# Factor 2: Proximidad a fantasmas
for ghost_state in ghost_states:
    ghost_pos = ghost_state.getPosition()
    ghost_distance = manhattanDistance(pacman_pos, ghost_pos)
    
    if ghost_state.scaredTimer > 0:
        # Si el fantasma está asustado, acercarse a él
        score += 50 / (ghost_distance + 1)
    else:
        # Si no está asustado, evitarlo
        if ghost_distance <= 2:
            score -= 200
```

#### 2.2 *Nuestras heurísticas*
Estas heurísticas son muy sencillas y simplemente tienen en cuenta casos simples, en primer lugar, si nos acercamos a la comida, ganamos más puntos, y en segundo lugar, evitaremos acercarnos a los fantasmas siempre y cuando no esten asustados, en ese caso, el objetivo será acercarse.

En nuestro caso, hemos querido modificar el comportamiento inical de la red y añadir nuevos factores, en primer lugar, tenemos el primer factor ligeramente modificado:
```py
if food:
    min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
    score += 5 / (min_food_distance + 1)
```
Aquí simplemente hemos querido dar más valor a la comida, haciendo que el agente la priorice, ya que es el único camino a la victoria.

El segundo factor si ha sufrido más cambios:
```py
for ghost_state in ghost_states:
    ghost_pos = ghost_state.getPosition()
    ghost_distance = manhattanDistance(pacman_pos, ghost_pos)
    
    if ghost_state.scaredTimer > 0: # Si el fantasma está asustado
        if ghost_distance <= 2:
            score -= 200 
        elif ghost_distance > ghost_state.scaredTimer:
            score -= 50 / (ghost_distance + 1) # Cuanto más lejos esté el fantasma asustado, mejor score
    else:
        # Si no está asustado, evitarlo
        score -= 100 / (ghost_distance + 1)
```

Aunque el objetivo del agente es obtener la mayor puntuación posible, hemos decidido que el agente tome decisiones más seguras, que aunque lleven a puntuaciones más bajas, llevan a un ratio de victorias mayor, y personalmente, consideramos que ganar con un puntaje medio es mejor que perder con uno alto (Además, ganar da +500 puntos y perder -500). Para conseguir esto, hemos realizado la siguiente lógica: Si el fantasma está asustado, no queremos que comerlo sea una prioridad, por tanto vamos a perjudicar mucho si Pac-Man está cerca de él, además, queremos que mantenga una distancia segura, es decir, si el fantasma tiene el estado asustado 5 turnos más, Pac-Man debe mantenerse aproximadamente a esa misma distancia. Por otro lado si el fantasma no está asustado, seguimos evitandolo, pero a diferencia del factor original, hemos usado un gradiente, para penalizar más o menos según la distancia a la que se encuentra.

A partir de ahora, continuamos con los factores añadidos por nosotros, primero, el tercer factor, priorizar comida cuando los fantasmas están asustados:
```py
times = []
for ghost_state in ghost_states:
    times.append(ghost_state.scaredTimer)

if all(times) and food:
    min_food_distance = min(manhattanDistance(pacman_pos, food_pos) for food_pos in food)
    score += 200 / (min_food_distance + 1)
```

En primer lugar, comprobamos que todos los fantasmas estén asustados, ya que si alguno no lo está, está heurística daría problemas, ya que compensaría totalmente la penalización por cercanía a los fantasmas, haciendo que realice moviemientos arriesgados, lo cual es justamente lo contrario a nuestro objetivo. Entonces, si todos los fantasmas están asustados, vamos a beneficiar en gran medida el acercarnos a la comida, ya que este tiempo es perfecto para llegar a la victoria.

A continuación, el cuarto factor, la estrategía defensiva de comer capsulas si percibimos peligro:
```py
if capsules:
    min_capsule_distance = min(manhattanDistance(pacman_pos, capsule_pos) for capsule_pos in capsules)
    fantasmas_cerca = False
    for ghost_state in ghost_states:
        ghost_pos = ghost_state.getPosition()
        ghost_distance = manhattanDistance(pacman_pos, ghost_pos)

        if ghost_distance <= 4 and ghost_state.scaredTimer == 0:
            fantasmas_cerca = True
            break
        
    if fantasmas_cerca:
        if min_capsule_distance <= 3:
            score += 50 / (min_capsule_distance + 1)
    else:
        if min_capsule_distance <= 3:
            score += 5 / (min_capsule_distance + 1)
```

En esta heurística, comprobamos si algun fantasma está cerca, si es así, hacemos que Pac-Man se acerque a las capsulas para evitar entrar en peligro, en cambio, si no hay ningún fantasma cercano (o están asustados), la capsula no es una prioridad, aunque aumentamos ligeramente la puntuación ya que el agente se encuentra cerca de comida, lo cual es bueno.

En esta práctica se nos ha propuesto implementar unas instrucciones y estrategias para mejorar una red neuronal para que juegue al juego de Pac-Man de forma automática y eficiente, es decir, intentando que gane y consiga bastantes puntos.

Parte del trabajo ha consistido en entrenar la red neuronal, por lo que hemos jugado muchas partidas y las hemos guardado para que la red aprenda de nuestros pasos. Como algunas de las partidas que se guardaban eran malas, solamente hicimos que la red aprendiera de nuestras partidas buenas: las que habíamos ganados o en las que habíamos conseguido mucha puntuación al menos.

De esta forma, la máquina imitaría solamente patrones que siguiéramos en partidas que habían funcionado, haciendo que su entrenamiento haga que las partidas hechas por la máquina se parezcan de cierta manera a nuestras partidas que habían salido bien, por lo que se acercaría también a la victoria.

Como la red neuronal por sí sola no es suficiente, hemos tenido que implementar ciertas heurísticas. Ya venían dos implementadas, de las que hemos tenido que cambiar una: proximidad a fantasmas. En la original, cuando un fantasma estaba en su modo "normal", es decir, persiguiéndote, tú huías. Para hacer esto hacíamos que se restara puntuación mediante más cerca estabas del fantasma, así que el Pac-Man acaba yéndose a donde más puntuación haya (más lejos del fantasma). Esto no lo hemos cambiado, el problema es que cuando el fantasma está asustado íbamos a por él, pero son varios segundos en los que está vulnerable y por tanto no te puede hacer perder. Si nos lo comemos nos da algunos puntos pero estaremos en peligro más rápido. Lo que hacemos para cambiar esto es alejarse siempre del fantasma, esté asustado o no.
Nos fijamos en que si el fantasma asustado está relativamente cerca, Pac-Man puede preferir quedarse quieto a comer comida más cercana al fantasma. Para hacer que se acerque a esa comida, haremos que Pac-Man pueda acercarse a un fantasma asustado siempre que ocurran dos condiciones. La primera es que el fantasma esté a más de una posición de distancia, es decir, que se asuste a partir de verlo a solo una posición, ya que no queremos que se lo coma directamente pero sí lo que hay alrededor. La segunda condición es que si le queda poco tiempo de estar asustado al fantasma Pac-Man no se acerque incluso a distancias más grandes como puede ser una distancia de tres o cuatro, ya que así está preparado para cuando el fantasma le persiga.

La última heurística implementada por nosotros consiste en que comer cápsulas (la comida grande que hace que un fantasma esté asustado) dé muchos puntos cuando un fantasma está cerca, pero no tantos cuando el fantasma no lo esté, haciendo que sea mejor comer la cápsula cuando Pac-Man esté en peligro para que se aproveche más, pero si se la come aunque los fantasmas estén asustados o lejos, no estaría mal aunque no puntúe tanto. 

Para entrenar la red neuronal comenzamos jugando muchas partidas y guardándolas todas. Como no somos nuevos jugando al Pac-Man, no lo hacíamos demasiado mal pero era imposible ganar todas las partidas, de hecho en algunas se nos complicaba al inicio por despistes y perdíamos muy rápido. Como la red neuronal aprendía de todas las partidas (buenas y malas), a veces simplemente hacía movimientos que no llevaban a nada. En un intento de mejorarla, decidimos solamente guardar las partidas que salían muy bien o hasta que ganábamos. Ahí nos dimos cuenta de que a veces repetía movimientos que hacíamos muy seguidos, como por ejemplo empezar recorriendo la parte izquierda del mapa e ir subiendo por esa zona, pero había otro problema, y es que según avanzaba la partida nuestros movimientos se hacían más imprecisos, ya que era fácil repetir un patrón inicial pero luego -al igual que nosotros- la red se descontrolaba. Para fomentar el primer patrón correcto, copiamos unas pocas veces una de las partidas ganadas por si se centraba en replicar más los movimientos de una partida buena, y funcionaba muy bien

Aun con todo lo que habíamos hecho en la primera sesión, ninguna partida de las que probamos conseguía ganar, aunque algunas lo hacían más o menos bien, obteniendo . En total hemos usado 16 partidas originales para entrenar la red.

Para poder comparar y comentar nuestros resultados, tenemos que fijarnos en las tablas de la tarea 4, donde podemos ver diferencias claras entre las tres configuraciones que hemos usado: la red neuronal sin modificaciones, la red neuronal junto a las heurísticas y finalmente todo lo mencionado con Alpha-Beta.

Para cada laberinto y configuración, hemos usado las semillas de la 0 a la 9 (las primeras 10) para ver el comportamiento de la configuración en cada uno de los mapas, y así poder comparar de forma justa, ya que no tendría sentido comparar la semilla 0 con la 8 por ejemplo, tendrá que ser 0 con 0, 1 con 1...

Para empezar, nos dimos cuenta de que con la red neuronal sin ni siquiera heurísticas u otras modificaciones hechas por nosotros, es decir, como se nos dio originalmente, no ganaba ninguna partida en ninguno de los dos laberintos y todos los resultados eran muy bajos. No nos extrañó ya que alguna iba más o menos bien, pero la mayoría de partidas terminaban muy rápidamente. Funcionaba ligeramente mejor en el mapa que no se nos dio inicialmente que en el nuevo mapa, pero pudo haber sido casualidad ya que en ambos la media era aproximadamente de -300. 

Para continuar probamos a hacerlo con nuestras heurísticas y los resultados seguían siendo muy negativos, lo que de primeras nos hizo pensar si podía haber algún fallo, pero realmente las heurísticas las usábamos para tener una estrategia, y dicha estrategia no sería introducida hasta que incorporáramos Alpha-Beta, así que tenía sentido que aún no funcionara bien del todo. En este caso fue el segundo mapa el que iba un poco mejor que el primero, aun estando ambos en una puntuación media muy cercana a -300 y sin haber ganado ninguna partida.

Finalmente introducimos Alpha-Beta y resultó en un avance enorme para ambos mapas. Buscando el mejor resultado posible dentro de las capacidades de nuestros ordenadores y el tiempo que teníamos, aplicamos una profundidad de 5 y pasamos a empezar a ganar partidas, obteniendo 6/10 partidas ganadas en el primer mapa y 7/10 en el nuevo mapa, aun conservando la comparación de las mismas 10 semillas de antes. Las partidas que no ganaba seguían siendo buenas partidas, obteniendo puntuaciones entre 400 y 800 contando los puntos descontados al perder, y una media aproximadamente 1400 puntos para ambos mapas.